In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

save_dir = './checkpoints'
os.makedirs(save_dir, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

BATCH_SIZE = 512 
LEARNING_RATE = 0.4
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0001
EPOCHS = 50

class SimCLRTransform:
    def __init__(self):
        color_jitter = transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)
        self.train_transform = transforms.Compose([
            transforms.RandomResizedCrop(size=32, scale=(0.2, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomApply([color_jitter], p=0.8),
            transforms.RandomGrayscale(p=0.2),
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
        ])
    def __call__(self, x):
        return self.train_transform(x), self.train_transform(x)

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=SimCLRTransform())
train_loader = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)

class SimCLR_ResNet18(nn.Module):
    def __init__(self):
        super(SimCLR_ResNet18, self).__init__()
        base_model = torchvision.models.resnet18(weights=None)
        base_model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        base_model.maxpool = nn.Identity() 
        self.encoder = nn.Sequential(*list(base_model.children())[:-1])
        self.projector = nn.Sequential(
            nn.Linear(512, 512, bias=False),
            nn.Linear(512, 128, bias=False)
        )
    def forward(self, x):
        h = self.encoder(x).view(x.size(0), -1)
        z = self.projector(h) 
        return h, z

model = SimCLR_ResNet18().to(device)

class NTXentLoss(nn.Module):
    def __init__(self, temperature=0.1):
        super(NTXentLoss, self).__init__()
        self.temperature = temperature
        self.criterion = nn.CrossEntropyLoss(reduction="sum")
    def forward(self, z_i, z_j):
        batch_size = z_i.size(0)
        z_i, z_j = F.normalize(z_i, dim=1), F.normalize(z_j, dim=1)
        representations = torch.cat([z_i, z_j], dim=0)
        similarity_matrix = torch.matmul(representations, representations.T)
        labels = torch.cat([torch.arange(batch_size) + batch_size, torch.arange(batch_size)]).to(device)
        mask = torch.eye(2 * batch_size, dtype=torch.bool).to(device)
        similarity_matrix.masked_fill_(mask, -9e15)
        logits = similarity_matrix / self.temperature
        return self.criterion(logits, labels) / (2 * batch_size)

criterion = NTXentLoss()
optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

for epoch in range(1, EPOCHS + 1):
    model.train(); running_loss = 0.0
    for (view1, view2), _ in train_loader:
        view1, view2 = view1.to(device), view2.to(device)
        optimizer.zero_grad()
        _, z_i = model(view1); _, z_j = model(view2)
        loss = criterion(z_i, z_j); loss.backward(); optimizer.step()
        running_loss += loss.item()
    scheduler.step()
    print(f"Epoch [{epoch}/{EPOCHS}] Loss: {running_loss / len(train_loader):.4f}")
    if epoch % 10 == 0:
        torch.save(model.state_dict(), os.path.join(save_dir, f'simclr_resnet18_epoch_{epoch}.pth'))
print("Training complete.")